# 17b Does a larger thinking model judge stale evidence better than the 1.7B model and the trained checkers? (Google Colab, GPU)

Notebook 17 scored four small trained checkers and a prompted Qwen3-1.7B (thinking off) on pairs where the claim stays the same and the evidence changes: 300 VitaminC Wikipedia flip pairs and 118 Meridian superseded-policy pairs. On the Meridian pairs the 1.7B judge ranked the current text higher in 112 of 118, MiniCheck in 111, the cross-encoder in 107. This notebook asks the same question of **TwIL-8B**, an 8.8-billion-parameter model that thinks before it answers (webAI's fine-tune of IBM Granite 4.2 8B).

**What it measures.** For each claim and its evidence, the model writes a reasoning trace and a final Yes or No. The notebook reports accuracy with Wilson intervals, the share of pairs where both sides were judged right, an exact McNemar test against the 1.7B judge on the same pairs, and the cost: **tokens written per claim** and **seconds per claim** (batch time divided by batch size, so a throughput figure on one T4, not the latency of one request). The small checkers take about 0.35 s per claim on a laptop CPU (notebook 17), so the question is what the extra reading buys and what it costs.

**What differs from notebook 17.** The 1.7B judge gave a probability from one forward pass with thinking off. This model gives a verdict after a sampled reasoning trace (temperature 1.0, top-p 0.95, the card's settings), so a rerun with another seed can change a verdict. The prompt also asks it to think first. The comparison therefore uses verdicts, not rankings: a pair counts as right when the current text is judged supported and the stale text is not.

**License.** The weights carry the webAI Non-Commercial License 1.0 (non-commercial research and educational use). This notebook is research for the book and downloads the weights from Hugging Face at run time; nothing else in this repository depends on them.

**Runtime.** Colab, Runtime, Change runtime type, T4 GPU. The weights load in 4-bit (about 5.5 GB) on a GPU with less than 40 GB; the card's scores are for bf16, so 4-bit can differ slightly. The probe cell times 8 claims and prints the projected time for the full run before you start it. Nothing here is a result until you run it.

In [ ]:
SMOKE = False   # True: quick check of the code with a tiny stand-in model on CPU. Leave False for the real run.
import os
SMOKE = SMOKE or os.environ.get("SMOKE") == "1"
if not SMOKE:
    !pip -q install transformers==4.57.6 bitsandbytes accelerate 2>&1 | tail -n 1
import json, math, re, time, random, collections, urllib.request
import numpy as np, torch
from scipy.stats import binomtest
dev = "cuda" if torch.cuda.is_available() else "cpu"
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if dev == "cuda" else 0
print("torch", torch.__version__, "| device:", torch.cuda.get_device_name(0) if dev == "cuda" else "cpu", f"| {gpu_gb:.0f} GB")
import transformers, sys
_gpu = torch.cuda.is_available()
print("preflight | python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU:", (torch.cuda.get_device_name(0) + f", {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB") if _gpu else "none", flush=True)
if not SMOKE and not _gpu:
    raise SystemExit("No GPU. In Colab: Runtime, Change runtime type, choose a GPU (T4 is enough), then run all again. CPU and TPU runtimes are not supported.")

MODEL    = "Qwen/Qwen3-0.6B" if SMOKE else "webAI-Official/TwIL-8B"
N_MER    = 3 if SMOKE else 118      # Meridian pairs (118 = all of them)
N_VITC   = 2 if SMOKE else 300      # VitaminC flip pairs, the first N_VITC of the shuffled list notebook 17 used
MAX_NEW  = 48 if SMOKE else 4096    # tokens per answer; an answer that hits this is "cut" and counts as wrong
BATCH    = 4 if SMOKE else 8
SEED     = 0
RAW = "https://raw.githubusercontent.com/benman1/deterministic-ai-workshop/main/data/results/17_grounding_checkers/"
LOCAL = os.environ.get("LOCAL_DATA", "")   # used only by the smoke test

**Goal.** Load the same pairs notebook 17 scored, in the same order, and the 1.7B judge's saved scores for them. VitaminC (Schuster et al., 2021; CC BY-SA 3.0) comes from Hugging Face; the Meridian pairs and the saved 1.7B scores come from this repository on GitHub, with an upload prompt if GitHub does not have them yet. A flip pair is one claim with evidence that supports it and evidence that refutes it.

In [ ]:
def fetch(name, url):
    if LOCAL and os.path.exists(os.path.join(LOCAL, name)):
        return open(os.path.join(LOCAL, name)).read()
    try:
        return urllib.request.urlopen(url, timeout=120).read().decode()
    except Exception as e:
        print("could not fetch", name, "->", repr(e)[:80])
        from google.colab import files
        print("upload", name); up = files.upload(); return up[name].decode()

rows = [json.loads(l) for l in fetch("vitc_test.jsonl", "https://huggingface.co/datasets/tals/vitaminc/resolve/main/test.jsonl").splitlines()]
g = collections.defaultdict(list)
for r in rows:
    if r["revision_type"] == "real": g[r["case_id"]].append(r)
pairs = []
for cid, v in g.items():
    by = {}
    for r in v:
        if r["label"] in ("SUPPORTS", "REFUTES"): by.setdefault((r["claim"], r["label"]), r)
    for c in sorted({c for c, _ in by}):
        if (c, "SUPPORTS") in by and (c, "REFUTES") in by:
            pairs.append((cid, by[(c, "SUPPORTS")], by[(c, "REFUTES")])); break
random.seed(0); random.shuffle(pairs)
vitc = [dict(set="vitaminc", pair=i, claim=s["claim"], ev={"S": s["evidence"], "R": r["evidence"]}) for i, (cid, s, r) in enumerate(pairs[:N_VITC])]

DROP = {76, 79}
mer_rows = [json.loads(l) for l in fetch("meridian_pairs_llm_written.jsonl", RAW + "meridian_pairs_llm_written.jsonl").splitlines()]
mer_rows = [r for r in mer_rows if r["id"] not in DROP][:N_MER]
mer = [dict(set="meridian", pair=i, claim=r["claim"], ev={"S": r["current"], "R": r["stale"]}) for i, r in enumerate(mer_rows)]
if not SMOKE: assert len(mer) == 118, len(mer)

def saved_scores(name):
    out = {}
    for l in fetch(name, RAW + name).splitlines():
        d = json.loads(l); out[(d["pair"], d["side"])] = d["score"]
    return out
qwen = {"meridian": saved_scores("qwen3_1.7b_judge_meridian118.jsonl"), "vitaminc": saved_scores("qwen3_1.7b_judge_300.jsonl")}

items = []
for p in mer + vitc:
    for side in ("S", "R"):
        items.append(dict(set=p["set"], pair=p["pair"], side=side, y=1 if side == "S" else 0, claim=p["claim"], evidence=p["ev"][side]))
print(len(mer), "Meridian pairs,", len(vitc), "VitaminC pairs,", len(items), "claims to judge")

**Goal.** Load the model. On a GPU under 40 GB the weights load in 4-bit (NF4, float16 compute); otherwise in bfloat16. The tokenizer pads on the left so a batch of different prompt lengths can be generated together.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
if dev == "cuda" and gpu_gb < 40:
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=q, device_map="auto")
    precision = "4-bit NF4"
elif dev == "cuda":
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto"); precision = "bfloat16"
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32); precision = "float32 (smoke test)"
model.eval()
print(MODEL, precision, f"loaded in {time.time()-t0:.0f}s |", "memory", f"{torch.cuda.memory_allocated()/1e9:.1f} GB" if dev == "cuda" else "cpu")

**Goal.** Define the judge. The prompt is notebook 17's, with the last sentence changed to ask for reasoning first. The final verdict is the first Yes or No after the closing `</think>` tag. An answer with no closing tag (the budget ran out while thinking) or no Yes or No after it is **unparsed** and counts as wrong. The cell also tests the parser on five strings so a format surprise shows up here and not after an hour of generation.

In [ ]:
PROMPT = "Evidence: {e}\n\nClaim: {c}\n\nIs the claim supported by the evidence? Think it through, then answer with one word, Yes or No."
def build(it):
    msg = [{"role": "user", "content": PROMPT.format(e=it["evidence"], c=it["claim"])}]
    return tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)

def parse(text):
    """Return 'Yes', 'No' or None. Looks only after the last </think>."""
    if "</think>" not in text: return None
    tail = text.rsplit("</think>", 1)[1]
    m = re.search(r"\b(yes|no)\b", tail, flags=re.I)
    return m.group(1).capitalize() if m else None

for s, want in [("<think>hmm</think>\n\nYes.", "Yes"), ("<think>x</think> No, because the date changed.", "No"),
                ("<think>still thinking and the budget ends", None), ("<think>a</think>Not sure.", None), ("<think>a Yes b</think>\nno", "No")]:
    assert parse(s) == want, (s, parse(s), want)
print("parser self-test passed; example prompt:\n" + build(items[0])[:400])

**Goal.** Define the batch generation, then **probe**: judge 8 claims, print each verdict with its tokens and seconds, and project the time for all claims. Read these lines before starting the full run. Sampling uses the card's settings (temperature 1.0, top-p 0.95) with a fixed seed per batch.

In [ ]:
def run_batch(batch, seed):
    prompts = [build(it) for it in batch]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    torch.manual_seed(seed); t = time.time()
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=True, temperature=1.0, top_p=0.95, pad_token_id=tok.pad_token_id)
    sec = time.time() - t; res = []
    for i, it in enumerate(batch):
        gen = out[i, enc["input_ids"].shape[1]:].tolist()
        eos = [k for k, x in enumerate(gen) if x == tok.eos_token_id]
        cut = not eos; n = len(gen) if cut else eos[0] + 1
        text = tok.decode(gen[:n], skip_special_tokens=False)
        res.append(dict(set=it["set"], pair=it["pair"], side=it["side"], y=it["y"], verdict=parse(text), tokens=n, cut=cut, sec=sec / len(batch), text=text))
    return res

probe = run_batch(items[:8], SEED)
for r in probe:
    print(f"{r['set']:9s} pair {r['pair']:3d} {r['side']}  truth {'Yes' if r['y'] else 'No':3s}  verdict {str(r['verdict']):4s}  {r['tokens']:5d} tokens  cut {r['cut']}  {r['sec']:.1f} s")
sec_per = np.mean([r["sec"] for r in probe]); tok_per = np.mean([r["tokens"] for r in probe])
print(f"\nprobe: {tok_per:.0f} tokens and {sec_per:.1f} s per claim (batch time / batch size); projected for {len(items)} claims: {sec_per*len(items)/3600:.1f} hours")

**Goal.** Judge every claim. Results are appended to `twil_outputs.jsonl` after each batch, with one progress line per batch, so a stopped session loses at most one batch and rerunning this cell resumes where it stopped. If the projected time above is too long, lower `N_VITC` in the first cell: the Meridian pairs come first and are the question that matters most.

In [ ]:
OUTF = "twil_outputs.jsonl" if not SMOKE else "twil_outputs_smoke.jsonl"
done = {}
if os.path.exists(OUTF):
    for l in open(OUTF): d = json.loads(l); done[(d["set"], d["pair"], d["side"])] = d
todo = [it for it in items if (it["set"], it["pair"], it["side"]) not in done]
print(len(done), "already done,", len(todo), "to do")
t_all = time.time()
for b in range(0, len(todo), BATCH):
    batch = todo[b:b + BATCH]
    res = run_batch(batch, SEED + len(done) + b)
    with open(OUTF, "a") as f:
        for r in res: f.write(json.dumps(r) + "\n"); done[(r["set"], r["pair"], r["side"])] = r
    nd = len(done); cuts = sum(d["cut"] for d in done.values())
    ok = sum((d["verdict"] == "Yes") == bool(d["y"]) and d["verdict"] is not None for d in done.values())
    print(f"{nd}/{len(items)}  last batch {np.mean([r['tokens'] for r in res]):.0f} tokens/claim, {res[0]['sec']:.1f} s/claim  | running: right {ok}/{nd}, cut {cuts}  [{time.time()-t_all:.0f} s]", flush=True)

**Goal.** Score the run. For each set: accuracy over claims with a Wilson interval, the number of pairs where both sides were judged right (current supported, stale not), the same number for the saved 1.7B judge (score above 0.5 on both sides, from notebook 17's files), and an exact McNemar test on those pairs. The McNemar test looks only at pairs where exactly one of the two judges was right on both sides. The cost lines report tokens and seconds per claim and the cut and unparsed shares.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),) * 2
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - h) / d, (c + h) / d

def correct(r): return r["verdict"] is not None and (r["verdict"] == "Yes") == bool(r["y"])
summary = {"model": MODEL, "precision": precision, "max_new": MAX_NEW, "batch": BATCH, "seed": SEED, "sets": {}}
for name, n_pairs in (("meridian", len(mer)), ("vitaminc", len(vitc))):
    recs = [d for d in done.values() if d["set"] == name]
    if not recs: continue
    by = collections.defaultdict(dict)
    for d in recs: by[d["pair"]][d["side"]] = d
    full = [p for p, s in by.items() if len(s) == 2]
    k = sum(correct(d) for d in recs); n = len(recs); lo, hi = wilson(k, n)
    both = [p for p in full if correct(by[p]["S"]) and correct(by[p]["R"])]
    qboth = [p for p in full if qwen[name][(p, "S")] > 0.5 and qwen[name][(p, "R")] <= 0.5]
    a = sum(1 for p in full if p in both and p not in qboth); b = sum(1 for p in full if p not in both and p in qboth)
    pv = binomtest(a, a + b, 0.5).pvalue if a + b else 1.0
    kb, nb = len(both), len(full); blo, bhi = wilson(kb, nb); kq = len(qboth); qlo, qhi = wilson(kq, nb)
    toks = np.array([d["tokens"] for d in recs]); secs = np.array([d["sec"] for d in recs])
    cut = sum(d["cut"] for d in recs); unp = sum(d["verdict"] is None and not d["cut"] for d in recs)
    print(f"{name}: {len(recs)} claims judged ({len(full)} full pairs)")
    print(f"  claim accuracy {k}/{n} = {k/n:.3f} (Wilson {lo:.3f} to {hi:.3f})")
    print(f"  pairs with both sides right: TwIL {kb}/{nb} = {kb/nb:.3f} (Wilson {blo:.3f} to {bhi:.3f}); Qwen3-1.7B {kq}/{nb} = {kq/nb:.3f} (Wilson {qlo:.3f} to {qhi:.3f}); only TwIL {a}, only Qwen {b}, exact McNemar p = {pv:.3g}")
    print(f"  cost per claim: median {np.median(toks):.0f} tokens (95th percentile {np.percentile(toks,95):.0f}), mean {secs.mean():.1f} s on this machine; cut at {MAX_NEW} tokens: {cut}/{len(recs)}; finished without a Yes or No: {unp}")
    summary["sets"][name] = dict(claims=len(recs), pairs=nb, claim_correct=k, claim_wilson=[lo, hi], pairs_both_right=kb, pairs_both_wilson=[blo, bhi],
                                 qwen_pairs_both_right=kq, only_twil=a, only_qwen=b, mcnemar_p=pv, tokens_median=float(np.median(toks)),
                                 tokens_p95=float(np.percentile(toks, 95)), sec_mean=float(secs.mean()), cut=cut, unparsed=unp)
json.dump(summary, open("17b_twil_results.json" if not SMOKE else "17b_twil_results_smoke.json", "w"), indent=1)
print("\nwrote 17b_twil_results.json (download it from the Colab file browser together with twil_outputs.jsonl)")

**What to read from this.** Two numbers decide whether the larger model earns a place in the book's judge section. First, the interval for pairs with both sides right: if TwIL's lower end sits above the 1.7B judge's point value on Meridian, the extra reading helps; if the intervals overlap, as they did among the small checkers, it does not. Second, the cost: the median tokens per claim times the seconds per token is what each decision costs, against about 0.35 s on a laptop CPU for MiniCheck. Report both in the same sentence.

**Limits.** One sampled run per claim at temperature 1.0, so verdicts are not reproducible without the seed; 4-bit weights when run on a T4, where the card's scores are for bf16; the Meridian pairs were written by a language model, and the date-rule baseline that reads the effective dates gets all of them right by construction; a claim cut at the token budget is counted wrong, which charges the model for length.